# Artoria Multi-Style LoRA Training

Clean, output-free reconstruction of the verified four-style Google Colab workflow. See `TRAINING_PROVENANCE.md` for the original notebook and artifact hashes. This notebook needs a Colab Kaggle API token named `KAGGLE_API_TOKEN`.

In [ ]:
from google.colab import drive, userdata
import glob
import os

drive.mount('/content/drive')
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
!pip -q install kaggle

DATASET_SLUG = 'h7alasaleh/artoria-dataset'
DRIVE_ROOT = '/content/drive/MyDrive/datasets/artoria'
LOCAL_DATA = '/content/data'
os.makedirs(DRIVE_ROOT, exist_ok=True)
if not glob.glob(f'{DRIVE_ROOT}/*.zip'):
    !kaggle datasets download -d {DATASET_SLUG} -p {DRIVE_ROOT} --force
zip_path = glob.glob(f'{DRIVE_ROOT}/*.zip')[0]
if not os.path.exists(os.path.join(LOCAL_DATA, 'CleanedDataSet')):
    os.makedirs(LOCAL_DATA, exist_ok=True)
    !unzip -q {zip_path} -d {LOCAL_DATA}
BASE_DATASET_DIR = LOCAL_DATA


In [ ]:
# Pin the reconstructable Diffusers state rather than installing a moving main branch.
DIFFUSERS_COMMIT = 'd0c9cbad28d7d3bba28db94622e13500c4179075'
!pip -q install git+https://github.com/huggingface/diffusers.git@{DIFFUSERS_COMMIT}
!pip -q install transformers accelerate peft bitsandbytes datasets xformers
!wget -q -O train_text_to_image_lora.py https://raw.githubusercontent.com/huggingface/diffusers/{DIFFUSERS_COMMIT}/examples/text_to_image/train_text_to_image_lora.py


In [ ]:
import json
import shutil

STYLES = [
    {'name': 'ukiyo_e', 'source': 'CUkiyo_e', 'prompt': 'a ukiyo-e style painting, high quality, authentic woodblock print'},
    {'name': 'cubism', 'source': 'CCubism', 'prompt': 'a cubism style painting, geometric shapes, abstract, high quality'},
    {'name': 'pop_art', 'source': 'CPop_Art', 'prompt': 'a pop art style illustration, bold colors, comic style, high quality'},
    {'name': 'post_impressionism', 'source': 'CPost_Impressionism', 'prompt': 'a post-impressionism style painting, expressive brushstrokes, vivid colors'},
]
SOURCE_ROOT = f'{BASE_DATASET_DIR}/CleanedDataSet'
TRAIN_ROOT = '/content/working/train_data'
os.makedirs(TRAIN_ROOT, exist_ok=True)

for style in STYLES:
    source = os.path.join(SOURCE_ROOT, style['source'])
    target = os.path.join(TRAIN_ROOT, style['name'])
    os.makedirs(target, exist_ok=True)
    images = [p for p in glob.glob(f'{source}/*.*') if p.lower().endswith(('.png', '.jpg', '.jpeg'))]
    with open(os.path.join(target, 'metadata.jsonl'), 'w', encoding='utf-8') as handle:
        for image_path in images:
            name = os.path.basename(image_path)
            shutil.copy2(image_path, os.path.join(target, name))
            handle.write(json.dumps({'file_name': name, 'text': style['prompt']}) + '\n')
    print(style['name'], len(images))


In [ ]:
from pathlib import Path
import subprocess
import sys

Path('config.yaml').write_text('''compute_environment: LOCAL_MACHINE
distributed_type: NO
num_processes: 1
gpu_ids: 0
mixed_precision: fp16
''')

for style in STYLES:
    style_id = style['name']
    output_dir = f'/content/working/lora-output-{style_id.replace("_", "-")}'
    command = [
        'accelerate', 'launch', '--config_file', 'config.yaml', 'train_text_to_image_lora.py',
        '--pretrained_model_name_or_path=runwayml/stable-diffusion-v1-5',
        f'--train_data_dir={TRAIN_ROOT}/{style_id}', '--resolution=512', '--center_crop', '--random_flip',
        '--train_batch_size=1', '--gradient_accumulation_steps=4', '--max_train_steps=1000',
        '--learning_rate=1e-4', '--max_grad_norm=1', '--lr_scheduler=cosine', '--lr_warmup_steps=0',
        f'--output_dir={output_dir}', '--checkpointing_steps=500', '--seed=42',
        '--mixed_precision=fp16', '--caption_column=text',
    ]
    subprocess.run(command, check=True)


In [ ]:
# Create the same final-weight-only archive layout as the verified run.
from google.colab import files
import zipfile

STYLE_MAP = {'Ukiyo-e': 'ukiyo-e', 'Cubism': 'cubism', 'Pop Art': 'pop-art', 'Post-Impressionism': 'post-impressionism'}
archive_path = '/content/final_lora_weights_only.zip'
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for label, style_id in STYLE_MAP.items():
        weight = f'/content/working/lora-output-{style_id}/pytorch_lora_weights.safetensors'
        archive.write(weight, f'{label.replace(" ", "_")}/pytorch_lora_weights.safetensors')
files.download(archive_path)
